# 03 — Cleaning, target construction, feature engineering, normalization and feature selection
This notebook shows every preprocessing step and demonstrates the leakage controls.

**Forecast protocol.** Issue time = 18:00 Asia/Almaty on day *d*; target = mean PM2.5 over local day *d+1*
(≥ 20 valid hours). Records are labelled by the start of their hour, so at 18:00 the newest usable record is
the one labelled 17:00.

In [1]:
# --- Setup: locate the project root and import the package --------------------
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
from IPython.display import Image, display, Markdown
from air_quality.config import load_config, setup_logging
cfg = load_config(ROOT / "config.yaml"); setup_logging(cfg, "notebooks")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
FIG = cfg.path("figures_dir"); MET = cfg.path("metrics_dir")
print("Project root:", ROOT)

Project root: /Users/dannieey/Desktop/ML_final


In [2]:
if not cfg.path("raw_csv").exists():
    raise FileNotFoundError("Raw data not found. Run: python -m air_quality.cli download  (requires OPENWEATHER_API_KEY in .env)")

## 1. Cleaning (global, split-independent operations)

In [3]:
from air_quality.preprocessing import clean_hourly
raw = pd.read_csv(cfg.path("raw_csv"))
clean, rep = clean_hourly(raw, cfg.tz)
for s in rep["steps"]:
    print(s)
print("\nGaps:", rep["gaps"]); print("Extremes:", rep["extremes"])

/Users/dannieey/Desktop/ML_final/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


{'step': 'drop_invalid_timestamps', 'removed': 0}
{'step': 'drop_future_timestamps', 'removed': 0}
{'step': 'sort_chronologically', 'was_sorted': True}
{'step': 'remove_duplicates', 'exact_duplicates_removed': 0, 'conflicting_duplicate_rows': 0, 'conflicting_removed_keep_first': 0, 'reason': 'one observation per hourly timestamp; first occurrence kept'}
{'step': 'invalidate_impossible_values', 'set_to_nan': {'aqi': 0, 'co': 0, 'no': 0, 'no2': 2, 'o3': 1, 'so2': 0, 'pm2_5': 0, 'pm10': 1, 'nh3': 0}, 'rule': 'negative concentrations; AQI outside 1-5'}
{'step': 'reindex_complete_hourly_grid', 'grid_hours': 43969, 'missing_hours_inserted_as_nan': 696, 'off_grid_timestamps_floored': 0, 'note': 'missing hours are NOT interpolated'}

Gaps: {'n_gaps': 20, 'max_gap_hours': 120, 'gaps_over_24h': 6, 'gap_length_counts': {24: 14, 48: 5, 120: 1}}
Extremes: {'method': 'robust z-score of log1p(PM2.5) > 3.5 (flag only, retained)', 'n_flagged_hours': 0, 'pm25_p99': 35.588399999999965, 'pm25_max': 87.67}

In [4]:
q = json.loads(cfg.path("quality_report_json").read_text())
cmp = pd.DataFrame({"before": q["before"], "after": q["after"]}).loc[
    ["n_records", "n_unique_timestamps", "n_duplicated_timestamps", "coverage_pct", "missing_hours", "pm2_5_min", "pm2_5_max", "n_valid_days_min_20h"]]
display(cmp)

,before,after
n_records,43273,43273
n_unique_timestamps,43273,43273
n_duplicated_timestamps,0,0
coverage_pct,98.417,98.417
missing_hours,696,696
pm2_5_min,0.0,0.0
pm2_5_max,87.67,87.67
n_valid_days_min_20h,1783,1783


## 2. Daily target construction
$y(d) = \frac{1}{|H_{d+1}|}\sum_{h \in H_{d+1}} \mathrm{PM2.5}_h$, valid only if $|H_{d+1}| \geq 20$.

In [5]:
from air_quality.feature_engineering import daily_targets, build_dataset, compute_features, issue_time_for_date, FEATURE_SETS
t = daily_targets(clean, cfg["forecast"]["target_min_valid_hours"])
print(f"Local days: {len(t)} | valid targets: {int(t.target_is_valid.sum())} | excluded (<20 h): {int((~t.target_is_valid).sum())}")
display(t.target_valid_hours.value_counts().sort_index().rename("days").to_frame().T)

Local days: 1823 | valid targets: 1783 | excluded (<20 h): 40


target_valid_hours,6,7,17,18,24,25
days,9,11,11,9,1782,1


## 3. Features computed at the issue time — worked example

In [6]:
ds = pd.read_csv(cfg.path("features_csv"))
ex_date = ds[ds.eligible & (ds.split == "test")].issue_date.iloc[10]
issue = issue_time_for_date(ex_date, cfg.tz, cfg["forecast"]["issue_hour_local"])
fc = {**cfg["forecast"]}
row = compute_features(clean, [issue], fc, cfg["evaluation"]["heating_season"]).iloc[0]
cut = issue.tz_convert("UTC")
w = clean[(clean.timestamp_utc < cut) & (clean.timestamp_utc >= cut - pd.Timedelta(hours=24))]
print(f"Issue time {issue} -> target date {row.target_date}")
print(f"Window used for pm25_mean_24h: {w.timestamp_local.min()} .. {w.timestamp_local.max()} (newest label < issue)")
print(f"pm25_mean_24h = {row.pm25_mean_24h:.3f} | manual mean = {w.pm2_5.mean():.3f}")
display(row[FEATURE_SETS["full"]].to_frame("value").T)

Issue time 2025-01-10 18:00:00+05:00 -> target date 2025-01-11
Window used for pm25_mean_24h: 2025-01-09 18:00:00+05:00 .. 2025-01-10 17:00:00+05:00 (newest label < issue)
pm25_mean_24h = 1.686 | manual mean = 1.686


,target_dow,target_month,target_season,target_is_weekend,target_month_sin,target_month_cos,target_doy_sin,target_doy_cos,target_is_heating_season,pm25_last,pm25_lag_24h,pm25_lag_48h,pm25_lag_72h,pm10_last,no2_last,co_last,o3_last,so2_last,nh3_last,no_last,...,o3_lag_24h,pm25_mean_6h,pm25_mean_12h,pm25_mean_24h,pm25_mean_48h,pm25_mean_72h,pm25_std_24h,pm25_min_24h,pm25_max_24h,pm25_trend_24h,pm25_diff_24h,pm25_prev_day_mean,pm_ratio_24h,pm10_mean_24h,no2_mean_24h,co_mean_24h,o3_mean_24h,so2_mean_24h,nh3_mean_24h,no_mean_24h
value,5.0,1.0,0.0,1.0,0.0,1.0,0.171177,0.98524,1.0,2.74,2.44,5.76,9.97,3.13,7.45,263.69,66.52,8.34,0.07,0.01,...,73.67,2.151667,1.771667,1.685833,2.418542,3.25875,0.64438,0.81,2.79,0.002365,-1.465417,1.958333,0.619792,2.72,4.304167,254.65,70.096667,7.342083,0.117917,0.173333


## 4. Leakage checks (executed)
1. Overwrite every observation at/after the issue time with an absurd value — features must not change.
2. Overwrite the target day — features must not change.
3. Every row's newest record label + 1 h ≤ issue time.

In [7]:
g2 = clean.copy(); g2.loc[g2.timestamp_utc >= cut, ["pm2_5", "pm10", "no2", "co", "o3", "so2", "nh3", "no", "aqi"]] = 1e6
row2 = compute_features(g2, [issue], fc, cfg["evaluation"]["heating_season"]).iloc[0]
same = all((np.isnan(row[f]) and np.isnan(row2[f])) or np.isclose(row[f], row2[f]) for f in FEATURE_SETS["full_aqi"])
newest = pd.to_datetime(ds.meta_newest_record_utc, utc=True); iss = pd.to_datetime(ds.issue_time_utc, utc=True)
ok_rows = ((newest + pd.Timedelta(hours=1)) <= iss)[newest.notna()].all()
print("Check 1/2 (future values ignored):", "PASSED" if same else "FAILED")
print("Check 3 (all rows use only past records):", "PASSED" if ok_rows else "FAILED")

Check 1/2 (future values ignored): PASSED
Check 3 (all rows use only past records): PASSED


## 5. Feature sets used in the experiments

In [8]:
for k, v in FEATURE_SETS.items():
    print(f"{k:<9} {len(v):>3} features")
print("full_selected = full + in-pipeline selection (VarianceThreshold -> CorrelationFilter -> TopK f_regression)")
el = ds[ds.eligible]
display(el.groupby("split").agg(n=("target_pm25", "size"), first=("target_date", "min"), last=("target_date", "max"),
                                mean_target=("target_pm25", "mean")).round(2))

basic      10 features
lag        24 features
full       43 features
full_aqi   45 features
full_weather  61 features
full_weather_past  50 features
full_selected = full + in-pipeline selection (VarianceThreshold -> CorrelationFilter -> TopK f_regression)


,n,first,last,mean_target
split,,,,
test,346,2025-01-01,2025-12-31,5.11
train,1061,2021-01-01,2023-12-31,6.24
unused,5,2020-12-27,2020-12-31,8.11
validation,351,2024-01-01,2024-12-31,5.92


## 6. Training-only correlation analysis and feature selection

In [9]:
from air_quality.feature_selection import training_correlation_table, make_selector
from sklearn.impute import SimpleImputer
tr = el[el.split == "train"]
ct = training_correlation_table(tr, FEATURE_SETS["full"], "target_pm25")
display(ct.head(15).round(3))
Xtr = SimpleImputer(strategy="median", keep_empty_features=True).fit_transform(tr[FEATURE_SETS["full"]])
sel = make_selector(k=20).fit(Xtr, tr.target_pm25)
names = np.array(FEATURE_SETS["full"])
for _, step in sel.steps:
    print(f"{type(step).__name__:<18} keeps {step.get_support().sum():>3} of {len(names)}")
    names = names[step.get_support()]
print("Selected (k=20, illustration; k is tuned during training):", list(names))

,feature,pearson,spearman,n
9,pm25_last,0.520,0.620,1061
24,pm25_mean_6h,0.490,0.570,1061
25,pm25_mean_12h,0.484,0.519,1061
26,pm25_mean_24h,0.447,0.467,1061
30,pm25_min_24h,0.441,0.499,1061
13,pm10_last,0.441,0.603,1061
14,no2_last,0.434,0.491,1061
36,pm10_mean_24h,0.386,0.438,1061
31,pm25_max_24h,0.385,0.424,1061
17,so2_last,0.378,0.495,1061


VarianceThreshold  keeps  43 of 43
CorrelationFilter  keeps  40 of 43
TopKSelector       keeps  20 of 40
Selected (k=20, illustration; k is tuned during training): [np.str_('target_month_cos'), np.str_('pm25_last'), np.str_('pm10_last'), np.str_('no2_last'), np.str_('so2_last'), np.str_('nh3_last'), np.str_('no_last'), np.str_('pm25_mean_6h'), np.str_('pm25_mean_12h'), np.str_('pm25_mean_24h'), np.str_('pm25_mean_48h'), np.str_('pm25_mean_72h'), np.str_('pm25_std_24h'), np.str_('pm25_min_24h'), np.str_('pm25_prev_day_mean'), np.str_('pm10_mean_24h'), np.str_('no2_mean_24h'), np.str_('so2_mean_24h'), np.str_('nh3_mean_24h'), np.str_('no_mean_24h')]


/Users/dannieey/Desktop/ML_final/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/dannieey/Desktop/ML_final/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/dannieey/Desktop/ML_final/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 7. Normalization
`StandardScaler` is applied only inside the Ridge and SVR pipelines, after median imputation; both are fitted on training folds only. Tree models are scale-invariant and are not scaled.

In [10]:
from air_quality.models import build_pipeline
p = build_pipeline("svr").fit(tr[FEATURE_SETS["basic"]], tr.target_pm25)
print(p)
print("Scaler means learned from TRAIN only (first 5):", np.round(p.named_steps["scale"].mean_[:5], 3))

Pipeline(steps=[('impute',
                 SimpleImputer(keep_empty_features=True, strategy='median')),
                ('scale', StandardScaler()), ('model', SVR())])
Scaler means learned from TRAIN only (first 5): [ 2.999e+00  6.546e+00  1.525e+00  2.850e-01 -2.000e-03]
